# Assignment 2b - Clustering using a Quadtree

### Make sure you read through this entire notebook before getting started with implementing the algorithm.


This notebook has the following structure:

- The first part introduces the concepts for that week. The theory is interleaved with small exercises, which have the goal of letting you practice the concepts that were just intruduced.
- At the end there are one or more larger exercises, which have the goal to test what you have learned earlier. These will be more difficult and will require more independent work than the exercises in the first part.


# Introduction to this template notebook

* This is a **personal** notebook.
* Make sure you work in a **copy** of `...-template.ipynb`,
**renamed** to `...-yourIDnr.ipynb`,
where `yourIDnr` is your TU/e identification number.

<div class="alert alert-danger" role="danger">
<h3>Integrity</h3>
<ul>
    <li>In this course you must act according to the rules of the TU/e code of scientific conduct.</li>
    <li>All the exercises and the graded assignments are to be executed individually and independently.</li>
    <li>You must not copy from the Internet, your friends, books... If you represent other people's work as your own, then that constitutes fraud and will be reported to the Examination Committee.</li>
    <li>Making your work available to others (complicity) also constitutes fraud.</li>
</ul>
</div>

You are expected to work with Python code in this notebook.

The locations where you should write your solutions can be recognized by
**marker lines**,
which look like this:

>`#//`
>    `BEGIN_TODO [Label]` `Description` `(n points)`
>
>`#//`
>    `END_TODO [Label]`

<div class="alert alert-warning" role="alert">Do NOT modify or delete these marker lines.  Keep them as they are.<br/>
<br/>
NEVER write code <i>outside</i> the marked blocks.
Such code cannot be evaluated.
</div>

Proceed in this notebook as follows:
* **Read** the text.
* **Fill in** your solutions between `BEGIN_TODO` and `END_TODO` marker lines.
* **Run** _all_ code cells (also the ones _without_ your code),
    _in linear order_ from the first code cell.

**Personalize your notebook**:
1. Copy the following three lines of code:

  ```python
  AUTHOR_NAME = 'Your Full Name'
  AUTHOR_ID_NR = '1234567'
  AUTHOR_DATE = 'YYYY-MM-DD'  # when notebook was first modified, e.g. '2020-02-26'
  ```

1. Paste them between the marker lines in the next code cell.
1. Fill in your _full name_, _identification number_, and the current _date_ as strings between quotes.
1. Run the code cell by putting the cursor there and typing **Control-Enter**.


In [8]:
#// BEGIN_TODO [Author] Name, Id.nr., Date, as strings (1 point)

AUTHOR_NAME = 'Catherine Kmiec'
AUTHOR_ID_NR = '2148021'
AUTHOR_DATE = '2026-06-14'

#// END_TODO [Author]

AUTHOR_NAME, AUTHOR_ID_NR, AUTHOR_DATE

('Catherine Kmiec', '2148021', '2026-06-14')

# Clustering using a Quadtree

For this assignment, you are expected to implement a quad tree that can be used by the DBscan clustering algorithm, as discussed in class. We expect you to implement four functions::
* Implement the ```__init__()``` function and have it create a correct quadtree.
* Implement the ```report_range()``` function. This function should, given a `point` and a `radius`, return all points in the quadtree that are within `radius` distance from `point`.
* Implement the ```count_range()``` function. This function should, given a `point` and a `radius`, return the number of points that lie within `radius` distance from `point`. Note that this can be computed faster than actually finding all the points.
* Implement the ```delete_point()``` function. This function should, given a `point`, remove this point from the quad tree. You may assume that the `point` that is requested to be removed, should still be in the quad tree.

In addition to this file, you are provided several python files:
* `dataset.py`. This file contains the `Dataset` datastructure, which is used to read the input from and write the output to file.
* `db_scan.py`. This file contains the `find_clustering()` function that uses your implementation of QuadTree to cluster the cluster points.
* `node.py`. This file contains the `Node` datastructure. You can use this object to represent a node in your quad tree.
* `point.py`.  This file contains the datastructures ```Point``` and ```ClusterPoint``` that you can use in your implementation of the two functions.
* `range_query.py`. This file contains the abstract datastructure `RangeQuery` of which `QuadTree` is an implementation. Moreover, this file also contains the `LinearQuery` implementation of `RangeQuery`. This implementation is relatively slow, but can be used in testing.
* `square.py`. This file contains the `Square` datastructure that can be used in creating the quad tree.

<b>Testing.</b> After (partially) implementing the four functions, you can run the <b>run</b> function at the bottom of this document. Just above this function, you are provided with the ```test_case_nr``` field. Changing this field to an integer value $x \in [0, 7]$ allows you to choose which testcase you would like to run. This then reads the file 0$x$.in from the <b>input</b> folder and provides the resulting clustering in 0$x$.out in the <b>output</b> folder.

<b>Visulisation.</b> To view the clustering you created, you can use the ```Visualizer.ipynb``` notebook that you are provided alongside this assignment. After executing the first few cells, you can simply execute the cell that corresponds to the testcase you want to view and a visualisation is provided.

<b>Handing in.</b> To verify your implementation, you are expected to hand in this file on Canvas. Here, we use the automated checking tool Momotor to check your implementation of the algorithm. After it has run through all the testcases (should take at most a couple minutes), you can see in the Momotor tab of the course how you scored. 
Note: you should only hand in _this_ file. The visualizer and other python files should not be handed in.

We move on to the actual coding part of this assignment. At the start we import some useful tools.

In [1]:
import time
import sys

from point import Point, ClusterPoint
from square import Square
from node import Node

from db_scan import *
from dataset import *
from range_query import RangeQuery  

assignment_nr = 2

You can use the below field to place your own functions.

In [ ]:
#// BEGIN_TODO [YOUR-OWN-FUNCTIONS]

#// END_TODO [YOUR-OWN-FUNCTIONS]

Below, you are expected to write the initialization function of the QuadTree, i.e. you have to build the tree, so that it can be used in queries later on. The other functions below are quite self-explanatory.

In [ ]:
class QuadTree(RangeQuery):
    """
    Implementation of RangeQuery using a QuadTree
    """

    def __init__(self, points):
        self.points = points

#// BEGIN_TODO [CREATE-QUAD-TREE]

        def insert(node, point):
            node.size += 1
            if not node.children:
                if node.point is None:
                    node.point = point
                else:
                    existing = node.point
                    node.point = None
                    for sq in node.square.splice_square():
                        node.children.append(Node(node, 0, sq, []))
                    for child in node.children:
                        if child.square.contains_point(existing):
                            insert(child, existing)
                            break
                    for child in node.children:
                        if child.square.contains_point(point):
                            insert(child, point)
                            break
            else:
                for child in node.children:
                    if child.square.contains_point(point):
                        insert(child, point)
                        break

        bounding_square = Square.get_bounding_square(self.points)
        root_node = Node(None, 0, bounding_square, [])
        for p in self.points:
            insert(root_node, p)

#// END_TODO [CREATE-QUAD-TREE]    
        self.root_node = root_node

In [ ]:
class QuadTree(QuadTree):
    def report_range(self, p, epsilon):
        """
        Returns the points that lie within range epsilon from p
        """

        neighbors = []
#// BEGIN_TODO [IMPLEMENT-FIND-NEIGHBORS]      

        eps_sqr = epsilon**2
        stack = [self.root_node]
        while stack:
            node = stack.pop()
            if node.size == 0:
                continue
            if node.square.is_contained_in_circle(p, epsilon):
                collect = [node]
                while collect:
                    n = collect.pop()
                    if n.point is not None:
                        neighbors.append(n.point)
                    collect.extend(n.children)
            elif node.square.overlaps_with_circle(p, epsilon):
                if not node.children:
                    if node.point is not None and p.sq_distance_to(node.point) <= eps_sqr:
                        neighbors.append(node.point)
                else:
                    stack.extend(node.children)

#// END_TODO [IMPLEMENT-FIND-NEIGHBORS]
        return neighbors

In [ ]:
class QuadTree(QuadTree):
    def count_range(self, p, epsilon):
        """
        Returns the number of points that lie within range epsilon from p
        """

        count = 0
#// BEGIN_TODO [IMPLEMENT-COUNT-RANGE]

        eps_sqr = epsilon**2
        stack = [self.root_node]
        while stack:
            node = stack.pop()
            if node.size == 0:
                continue
            if node.square.is_contained_in_circle(p, epsilon):
                count += node.size
            elif node.square.overlaps_with_circle(p, epsilon):
                if not node.children:
                    if node.point is not None and p.sq_distance_to(node.point) <= eps_sqr:
                        count += 1
                else:
                    stack.extend(node.children)

#// END_TODO [IMPLEMENT-COUNT-RANGE]
        return count

In [ ]:
class QuadTree(QuadTree):
    def delete_point(self, point):
        """
        Deletes point from this quad tree
        """
#// BEGIN_TODO [IMPLEMENT-DELETE-POINT]

        node = self.root_node
        while node.children:
            for child in node.children:
                if child.square.contains_point(point):
                    node = child
                    break
        node.point = None
        node.size -= 1
        par = node.parent
        while par is not None:
            par.size -= 1
            par = par.parent

#// END_TODO [IMPLEMENT-DELETE-POINT]

The below function uses your implementation of QuadTree to cluster the given set of input points.

In [3]:
def run(path_in, path_out):
    """
    Reads the input set, clusters the points and writes to output\\
    path_in     location of the input set\\
    path_out    location to print the output
    """

    # Read input from file
    try:
        with open(path_in, "r") as f:
            ds = Dataset.read_input(f)    
    except IOError:
        print("could not read input file:", path_in, file=sys.stderr)
        return
    
    # Initialize the quad tree
    query_obj = QuadTree(ds.cluster_points)
    
    # Initialize linear range query
    #query_obj = LinearQuery(ds.cluster_points)

    # Use the quad tree to create a clustering
    nr_clusters = find_clustering(ds.cluster_points, ds.epsilon, ds.min_pts, query_obj)
    
    try:
        ds.write_output(nr_clusters, path_out, assignment_nr)
    except IOError:
        print("could not write output to file:", path_out, file=sys.stderr)
    print("Cluster counter:", nr_clusters, file=sys.stderr)

Here, you can run your implementation. By changing ```test_case_nr```  to any value $x\in[0,7]$, you can select which testcase you would like to run.

In [4]:
# runtime parameters
test_case_nr = 0             # Here you specify which testcase you would like to run

if __name__ == "__main__":
    start = time.time()
    run("input/{:02d}.in".format(test_case_nr), "output/{:02d}.out".format(test_case_nr))
    end = time.time()
    print("Time taken:      {:.3f}s".format(end - start), file=sys.stderr)

TypeError: Can't instantiate abstract class QuadTree with abstract methods count_range, delete_point, report_range

That is all. At this point you should have all the information you should need to get started. If you have any questions you are free to ask the tutor overseeing the class. If you are experiencing issues with handing in your submission to Momotor, you can contact the responsible teaching assistant via email. Their email address can be found on Canvas.

Best of luck and happy clustering!

&copy; 2019-2020 - **TU/e** - Eindhoven University of Technology